Clase 010 · `010_importar_formatos` · El mismo conjunto de datos de ESOLMET,
entregado de cuatro maneras distintas. Cada archivo tiene un problema típico y
`read_csv` o `read_excel` tienen un argumento para resolverlo. En cada paso la
variable `esolmet` se vuelve a crear desde cero: es un recorrido por formatos,
no una cadena de transformaciones.

::: {.callout-important title="Datos"}
Los archivos `v2_nan.csv`, `v3.xlsx`, `v4_centinelas.csv` y
`v5_formato_mx.csv` no viajan en el repositorio; colócalos en
`clases/010_importar_formatos/data/001_raw/` (se comparten en clase).
:::

## Librerías

In [1]:
import pandas as pd

## `v2_nan.csv`: celdas vacías

Aquí las filas están completas (son más que en `v1_filas_faltantes.csv`), pero hay
celdas vacías. `read_csv` las reconoce como `NaN` sin pedirle nada. La evidencia está en
`info()`: la columna *Non-Null Count* es menor que el total de entradas en casi todas
las variables.

In [2]:
f = "../data/001_raw/v2_nan.csv"
esolmet = pd.read_csv(f,index_col=0,parse_dates=True)
esolmet.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 172433 entries, 2026-04-08 00:00:00 to 2026-08-05 18:04:00
Data columns (total 10 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   dhi       172433 non-null  float64
 1   dni       170762 non-null  float64
 2   ghi       165559 non-null  float64
 3   p_atm     143686 non-null  float64
 4   rain_acc  172433 non-null  float64
 5   rh        163235 non-null  float64
 6   tdb       167407 non-null  float64
 7   uv        162141 non-null  float64
 8   wd        168967 non-null  float64
 9   ws        169003 non-null  float64
dtypes: float64(10)
memory usage: 14.5 MB


## `v3.xlsx`: Excel con encabezado

`read_excel` lee hojas de cálculo; necesita `openpyxl` instalado en el proyecto
(`uv add openpyxl`). `skiprows=4` salta los cuatro renglones de encabezado que hay antes
de la tabla. No hace falta `parse_dates`: Excel guarda las fechas como fechas y `pandas`
construye el `DatetimeIndex` directamente. Este archivo solo trae abril de 2026.

In [7]:
f = "../data/001_raw/v3.xlsx"
esolmet = pd.read_excel(f,skiprows=4,index_col=0)
esolmet.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 33120 entries, 2026-04-08 00:00:00 to 2026-04-30 23:59:00
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   dhi       33120 non-null  float64
 1   dni       33120 non-null  float64
 2   ghi       33118 non-null  float64
 3   p_atm     15331 non-null  float64
 4   rain_acc  33120 non-null  float64
 5   rh        32461 non-null  float64
 6   tdb       32792 non-null  float64
 7   uv        33120 non-null  float64
 8   wd        33120 non-null  float64
 9   ws        33120 non-null  float64
dtypes: float64(10)
memory usage: 2.8 MB


## `v4_centinelas.csv`: valores centinela

En vez de dejar la celda vacía, este archivo marca los faltantes con *centinelas*:
`-999.9`, `NA`, `S/D`, `?` y `--`. Sin avisarle a `pandas`, los centinelas de texto
convierten la columna entera en `object` (texto) y `-999.9` pasa como un número válido
que arruina cualquier promedio. `na_values` recibe la lista y los convierte todos en
`NaN` al leer.

La línea comentada `esolmet.wd.astype(float)` es rastro de la exploración en clase: con
los centinelas de texto dentro, convertir `wd` a flotante fallaba. Con `na_values` ya no
hace falta.

In [25]:
f = "../data/001_raw/v4_centinelas.csv"
esolmet = pd.read_csv(f,index_col=0,parse_dates=True,na_values=[-999.9,"NA","S/D","?","--"])
# esolmet.wd.astype(float)
esolmet.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 172433 entries, 2026-04-08 00:00:00 to 2026-08-05 18:04:00
Data columns (total 10 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   dhi       172433 non-null  float64
 1   dni       172433 non-null  float64
 2   ghi       167269 non-null  float64
 3   p_atm     139369 non-null  float64
 4   rain_acc  172433 non-null  float64
 5   rh        166688 non-null  float64
 6   tdb       172433 non-null  float64
 7   uv        167187 non-null  float64
 8   wd        167309 non-null  float64
 9   ws        167436 non-null  float64
dtypes: float64(10)
memory usage: 14.5 MB


Confirmamos con `head()`: `p_atm` ya muestra `NaN` y `wd` es numérica.

In [16]:
esolmet.head()

,dhi,dni,ghi,p_atm,rain_acc,rh,tdb,uv,wd,ws
date,,,,,,,,,,
2026-04-08 00:00:00,0.0,0.0,0.0,NaN,0.0,54.05,23.56,0.0,4.604,1.088
2026-04-08 00:01:00,0.0,0.0,0.0,NaN,0.0,54.20,23.53,0.0,347.2,1.162
2026-04-08 00:02:00,0.0,0.0,0.0,NaN,0.0,54.41,23.49,0.0,0.157,1.125
2026-04-08 00:03:00,0.0,0.0,0.0,NaN,0.0,54.29,23.48,0.0,347.8,1.100
2026-04-08 00:04:00,0.0,0.0,0.0,NaN,0.0,54.57,23.44,0.0,347.9,1.088


## `v5_formato_mx.csv`: separador `;`, decimal `,` y fecha partida

El formato regional cambia dos cosas: las columnas se separan con `;` y los decimales
usan `,`. `delimiter` y `decimal` se lo dicen a `read_csv`. Además la fecha viene en dos
columnas, `fecha` y `hora`, en orden día/mes/año. Los pasos:

1. Pegar `fecha` y `hora` con un espacio en una columna nueva `date`.
2. Convertirla con `to_datetime(dayfirst=True)`, para que `05/08/2026` sea 5 de agosto
   y no 8 de mayo.
3. Hacerla índice con `set_index`.
4. Borrar `fecha` y `hora`, que ya no aportan nada.

La línea `# esolmet` quedó comentada a propósito: mostrar el `DataFrame` completo son
miles de filas de HTML dentro de la libreta; basta `info()` o `head()`.

In [48]:
f = "../data/001_raw/v5_formato_mx.csv"
esolmet = pd.read_csv(f,delimiter=";",decimal=",")
esolmet["date"] = esolmet.fecha + " " + esolmet.hora
# esolmet
esolmet.date = pd.to_datetime(esolmet.date,dayfirst=True)
esolmet.set_index("date",inplace=True)
del esolmet["fecha"]
del esolmet["hora"]
esolmet.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 172433 entries, 2026-04-08 00:00:00 to 2026-08-05 18:04:00
Data columns (total 10 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   dhi       172433 non-null  float64
 1   dni       172433 non-null  float64
 2   ghi       172431 non-null  float64
 3   p_atm     143686 non-null  float64
 4   rain_acc  172433 non-null  float64
 5   rh        168478 non-null  float64
 6   tdb       168800 non-null  float64
 7   uv        172433 non-null  float64
 8   wd        172433 non-null  float64
 9   ws        172433 non-null  float64
dtypes: float64(10)
memory usage: 14.5 MB


El resultado tiene la misma forma que los demás: `DatetimeIndex` y diez columnas numéricas.

In [49]:
esolmet.head()

,dhi,dni,ghi,p_atm,rain_acc,rh,tdb,uv,wd,ws
date,,,,,,,,,,
2026-04-08 00:00:00,0.0,0.0,0.0,NaN,0.0,54.05,23.56,0.0,4.604,1.088
2026-04-08 00:01:00,0.0,0.0,0.0,NaN,0.0,54.20,23.53,0.0,347.200,1.162
2026-04-08 00:02:00,0.0,0.0,0.0,NaN,0.0,54.41,23.49,0.0,0.157,1.125
2026-04-08 00:03:00,0.0,0.0,0.0,NaN,0.0,54.29,23.48,0.0,347.800,1.100
2026-04-08 00:04:00,0.0,0.0,0.0,NaN,0.0,54.57,23.44,0.0,347.900,1.088
